# De-identification Pipeline — nun.xlsx

**Before running**: `Runtime > Change runtime type > T4 GPU`

**Steps**:
1. Install dependencies
2. Mount Google Drive
3. Login to HuggingFace
4. Run de-identification on all 4 sheets (with checkpoint/resume)
5. Output saved to Google Drive

**What is redacted**: PERSON, PHONE, EMAIL, ADDRESS, NATIONAL_ID, HOSPITAL_IDS  
**Not redacted**: DATE (intentionally excluded)

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers torch sentencepiece pandas openpyxl tqdm huggingface_hub

In [ ]:
# Cell 2 — Mount Google Drive
# Upload nun.xlsx to your Google Drive first, then set the path below.
from google.colab import drive
drive.mount('/content/drive')

# *** EDIT THESE PATHS ***
INPUT_PATH      = '/content/drive/MyDrive/nun.xlsx'
OUTPUT_PATH     = '/content/drive/MyDrive/nun_deidentified.xlsx'
LOG_PATH        = '/content/drive/MyDrive/nun_detections_log.csv'
CHECKPOINT_PATH = '/content/drive/MyDrive/nun_checkpoint.json'

In [ ]:
# Cell 3 — HuggingFace login
# Get your token from https://huggingface.co/settings/tokens
from huggingface_hub import login
login()

In [ ]:
# Cell 4 — Configuration
import torch

MODEL_ID   = 'loolootech/no-name-ner-th'
DEVICE     = 0 if torch.cuda.is_available() else -1
BATCH_SIZE = 64 if DEVICE == 0 else 16

print(f'Device: {"GPU — " + torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU"}')
print(f'Batch size: {BATCH_SIZE}')

# DATE intentionally excluded
ENTITY_MAP = {
    'PERSON':       '[PERSON]',
    'PHONE':        '[PHONE]',
    'EMAIL':        '[EMAIL]',
    'ADDRESS':      '[ADDRESS]',
    'NATIONAL_ID':  '[NATIONAL_ID]',
    'HOSPITAL_IDS': '[HOSPITAL_IDS]',
}

SHEET_TEXT_COLUMNS = {
    'Admission_Record': [
        'PROGDESC', 'PROGLIST', 'SUBJECTIVE', 'OBJECTIVE', 'ASSESSMENT', 'PLAN',
    ],
    'IPT_PROGRESSNOTE': ['FOCUS', 'ACTION', 'RESPONSE'],
    'OPD_PROGRESSNOTE': [
        'CHIEFCOMPLAIN', 'HISTORY', 'PHYSICALEXAM', 'MANAGEMENT', 'PLAN',
        'RECOMMENDATION', 'NURSENOTE', 'PROGRESSNOTE', 'MANAGEPLAN',
        'CHIEFCOMPLAIN2', 'HISTORY2', 'PHYSICALEXAM2', 'MANAGEMENT2',
        'PLAN2', 'RECOMMENDATION2', 'PROGRESSNOTE2', 'MANAGEPLAN2',
    ],
    'Radiology': [
        'RESULT', 'RESULT2',
        'TEXTRSL1', 'TEXTRSL2', 'TEXTRSL3', 'TEXTRSL4',
        'TEXTRSL5', 'TEXTRSL6', 'TEXTRSL7', 'TEXTRSL8',
    ],
}

# Medical device eponyms + English verbs falsely flagged as PERSON
MEDICAL_EPONYM_BLOCKLIST = {
    'foley', 'hickman', 'broviac', 'dobhoff', 'salem', 'penrose',
    'jackson', 'pratt', 'levin', 'ewald', 'blakemore', 'sengstaken',
    'linton', 'minnesota', 'cantor', 'miller', 'abbott', 'mallory',
    'weiss', 'heimlich', 'trendelenburg', 'valsalva', 'romberg',
    'babinski', 'kernig', 'brudzinski', 'trousseau', 'chvostek',
    'retain', 'remove', 'insert', 'replace', 'change', 'monitor',
    'continue', 'start', 'stop', 'hold', 'apply', 'check', 'drain',
    'refer', 'follow', 'admit', 'discharge', 'transfer', 'consult',
    'review', 'adjust', 'control', 'manage', 'observe', 'maintain',
}

In [ ]:
# Cell 5 — Helper functions
import re, json
from pathlib import Path

HN_PATTERN = re.compile(r'(?i)\bHN\s*[\.:;]?\s*(\d{4,8}(?:/\d{2,4})?)')
NATIONAL_ID_PATTERN = re.compile(r'(?<!\d)\d{13}(?!\d)')

def regex_preprocess(text):
    if not text or not isinstance(text, str): return text
    text = HN_PATTERN.sub(r'HN [HOSPITAL_IDS]', text)
    text = NATIONAL_ID_PATTERN.sub('[NATIONAL_ID]', text)
    return text

def merge_entities(entities):
    if not entities: return []
    merged = []
    for ent in sorted(entities, key=lambda e: e['start']):
        label, start, end, score = ent['entity_group'], ent['start'], ent['end'], ent['score']
        if merged and merged[-1][2] == label and start <= merged[-1][1] + 1:
            ps, pe, pl, psc = merged[-1]
            merged[-1] = (ps, max(pe, end), pl, max(psc, score))
        else:
            merged.append((start, end, label, score))
    return merged

def is_medical_eponym(span, label):
    return label == 'PERSON' and span.strip().lower() in MEDICAL_EPONYM_BLOCKLIST

def anonymize_batch(ner_pipeline, texts):
    preprocessed = [regex_preprocess(t) if t and isinstance(t, str) else t for t in texts]
    valid_idx = [i for i, t in enumerate(preprocessed) if t and isinstance(t, str) and t.strip()]
    valid_texts = [preprocessed[i] for i in valid_idx]

    ner_results = {}
    if valid_texts:
        batch_out = ner_pipeline(valid_texts, batch_size=BATCH_SIZE)
        if not isinstance(batch_out[0], list):
            batch_out = [batch_out]
        for i, entities in zip(valid_idx, batch_out):
            ner_results[i] = entities

    results = []
    for i, text in enumerate(preprocessed):
        if i not in ner_results or not text or not isinstance(text, str) or not text.strip():
            results.append((texts[i], []))
            continue
        merged = merge_entities(ner_results[i])
        detected, anonymized = [], text
        for start, end, label, score in reversed(merged):
            span = text[start:end]
            if is_medical_eponym(span, label): continue
            token = ENTITY_MAP.get(label)
            if token is None: continue  # skip entities not in ENTITY_MAP (e.g. DATE)
            anonymized = anonymized[:start] + token + anonymized[end:]
            detected.append({'entity_type': label, 'start': start, 'end': end,
                             'score': round(score, 4), 'replacement': token})
        detected.reverse()
        results.append((anonymized, detected))
    return results

def load_checkpoint(path):
    p = Path(path)
    if p.exists():
        with open(p) as f: data = json.load(f)
        return set(tuple(x) for x in data.get('done', []))
    return set()

def save_checkpoint(done, path):
    with open(path, 'w') as f:
        json.dump({'done': [list(x) for x in done]}, f)

print('Helpers ready.')

In [ ]:
# Cell 6 — Load model and data
import pandas as pd
from transformers import pipeline
from tqdm.notebook import tqdm

print(f'Loading model: {MODEL_ID} ...')
ner = pipeline('token-classification', model=MODEL_ID,
               device=DEVICE, aggregation_strategy='simple')
print('Model loaded.')

done = load_checkpoint(CHECKPOINT_PATH)
if done:
    print(f'Resuming — {len(done)} column(s) already done.')

print(f'\nReading {INPUT_PATH} ...')
sheets = pd.read_excel(INPUT_PATH, engine='openpyxl', sheet_name=None)
for name, df in sheets.items():
    print(f'  {name}: {df.shape}')

In [ ]:
# Cell 7 — Run de-identification
total_entity_counts = {}
log_path = Path(LOG_PATH)
write_header = not log_path.exists()

for sheet_name, df in sheets.items():
    if sheet_name not in SHEET_TEXT_COLUMNS:
        print(f'Skipping sheet (no text cols defined): {sheet_name}')
        continue

    cols = [c for c in SHEET_TEXT_COLUMNS[sheet_name] if c in df.columns]
    print(f'\n=== {sheet_name} ({len(df):,} rows) — {cols} ===')

    for col in cols:
        key = (sheet_name, col)
        if key in done:
            print(f'  {col}: skipping (already done).')
            continue

        mask = df[col].notna() & (df[col].astype(str).str.strip() != '')
        indices = df.index[mask].tolist()
        print(f'  {col}: {len(indices):,} non-empty rows')

        col_detections = []
        for batch_start in tqdm(range(0, len(indices), BATCH_SIZE), desc=f'  {col}'):
            batch_idx = indices[batch_start: batch_start + BATCH_SIZE]
            batch_texts = [str(df.at[i, col]) for i in batch_idx]
            batch_results = anonymize_batch(ner, batch_texts)

            for i, (anonymized, detected) in zip(batch_idx, batch_results):
                df.at[i, col] = anonymized
                for det in detected:
                    total_entity_counts[det['entity_type']] = (
                        total_entity_counts.get(det['entity_type'], 0) + 1)
                    col_detections.append({
                        'sheet': sheet_name, 'row_index': i,
                        'row_number': i + 2, 'column': col,
                        **{k: v for k, v in det.items()},
                    })

        if col_detections:
            pd.DataFrame(col_detections).to_csv(
                log_path, mode='a', index=False, header=write_header)
            write_header = False

        done.add(key)
        save_checkpoint(done, CHECKPOINT_PATH)
        print(f'  {col}: done. Checkpoint saved.')

    sheets[sheet_name] = df

print('\nAll sheets processed.')

In [ ]:
# Cell 8 — Save output to Google Drive
print(f'Writing {OUTPUT_PATH} ...')
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
print('Saved.')

total = sum(total_entity_counts.values())
print(f'\nTotal entities redacted: {total:,}')
for etype, count in sorted(total_entity_counts.items(), key=lambda x: -x[1]):
    print(f'  {etype}: {count:,}')

Path(CHECKPOINT_PATH).unlink(missing_ok=True)
print('\nDone.')